# Exploring uv.lock structure

I've been using uv for a while and wanted to really understand what's inside the `uv.lock` file — every section, how it's organized, and how it guarantees reproducible installs. This notebook walks through a real lockfile cell by cell.

In [ ]:
import tomllib
from pathlib import Path
import pprint

# I'll generate a fresh uv.lock to work with — using a simple project
# with requests so the file is small enough to read end-to-end.
# In practice you'd point this at your own uv.lock.
LOCK_PATH = Path.cwd() / "uv.lock"

if not LOCK_PATH.exists():
    print(f"No uv.lock found at {LOCK_PATH}")
    print("Run `uv init --app demo && cd demo && uv add requests` first.")
else:
    with open(LOCK_PATH, "rb") as f:
        lock = tomllib.load(f)
    print(f"Loaded uv.lock — {len(lock.get('package', []))} packages resolved")

## Top-level keys

The lockfile has just three keys at the top: `version` (always 1 for now), `metadata`, and `package`. That's it. Everything else lives inside those.

In [ ]:
if LOCK_PATH.exists():
    for key in lock:
        val = lock[key]
        if isinstance(val, list):
            print(f"{key}: list[{len(val)} items]")
        elif isinstance(val, dict):
            print(f"{key}: dict with keys {list(val.keys())}")
        else:
            print(f"{key}: {val!r}")

### The `[metadata]` section

This is the control panel. It stores:
- **`manifest-hash`** — a SHA-256 of the `pyproject.toml` at lock time. If you edit pyproject.toml, uv spots the hash mismatch and knows it needs to re-resolve.
- **`requires-dist`** — the direct dependencies declared in pyproject.toml, including extras and markers.
- **`requires-python`** — the Python version range constraint.

The hash is how uv detects staleness without guessing.

In [ ]:
if LOCK_PATH.exists():
    meta = lock.get("metadata", {})
    print("manifest-hash:", meta.get("manifest-hash", "(none)")[:40] + "...")
    print("requires-python:", meta.get("requires-python", "(not set)"))
    reqs = meta.get("requires-dist", [])
    print(f"requires-dist: {len(reqs)} direct dependencies")
    for r in reqs:
        print(f"  - {r}")

## The `[[package]]` entries

Each resolved package — direct or transitive — gets one `[[package]]` entry. The key fields are:

| Field | What it holds |
|---|---|
| `name` | Package name (alphabetically sorted) |
| `version` | Exact resolved version |
| `source` | Where it came from (registry, git, path, url) |
| `dependencies` | List of `{name}` refs — links to other `[[package]]` entries |
| `sdist` / `wheels` | Integrity hashes for the distribution archives |

The alphabetical sort surprised me at first — I expected dependency-order. But it makes diffing easier: if you add one package, only its entry and the metadata hash change.

In [ ]:
if LOCK_PATH.exists():
    for pkg in lock.get("package", [])[:5]:
        name = pkg["name"]
        ver = pkg.get("version", "?")
        deps = [d["name"] for d in pkg.get("dependencies", [])]
        src_type = list(pkg.get("source", {}).keys())[0] if pkg.get("source") else "?"
        print(f"{name} {ver}  [source: {src_type}]")
        if deps:
            print(f"    depends on: {', '.join(deps)}")
        else:
            print(f"    (terminal dependency — no deps)")

## How it ensures reproducibility

Three mechanisms work together:

1. **Exact version pinning** — every version is a specific number like `2.32.3`, not `>=2.28`. No ambiguity.
2. **Content hashes** — each `[[package]]` entry stores `sha256` hashes for the source distribution and/or wheels. Before installing, uv verifies the downloaded archive matches the hash. This prevents both accidental corruption and supply-chain substitution attacks.
3. **The manifest hash** — the `[metadata]` section includes a hash of `pyproject.toml`. If someone changes the deps without running `uv lock`, uv detects the mismatch and refuses to sync until you re-resolve.

Together, these make `uv.lock` a single source of truth that can be checked into version control.

In [ ]:
if LOCK_PATH.exists():
    print("Hash coverage check:")
    with_hashes = 0
    no_hashes = 0
    for pkg in lock.get("package", []):
        if pkg.get("wheels") or pkg.get("sdist"):
            with_hashes += 1
        else:
            no_hashes += 1
    print(f"  Packages with hashes: {with_hashes}")
    print(f"  Packages without:     {no_hashes}")
    print(f"  Verification: {'PASS' if no_hashes == 0 else 'SOME MISSING'}")

## What I'd explore next

I want to compare lockfiles across platforms to see how uv handles platform-specific wheels. I also want to understand how the `source` field changes when you point to a git repo or a local path — and whether uv.lock can represent multiple registries at once.